# Agentic AI RAG Chatbot

Pipeline: PDF → Chunking → Hugging Face Embeddings → Pinecone → LangGraph → Grounded LLM (Ollama) → FastAPI

**Before you start**
1. Install [Ollama](https://ollama.com) and run `ollama pull llama3.2` in a terminal. Keep Ollama running.
2. Have a `.env` file with `PINECONE_API_KEY=...`
3. Run the cells **from top to bottom, once each**. After the install cell, restart the kernel once.

## 1. Install packages (run once, then restart the kernel)

In [ ]:
#%pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-pinecone langchain-ollama langgraph pinecone pypdf fonttools sentence-transformers fastapi uvicorn python-dotenv requests nest-asyncio

## 2. Imports

In [1]:
import os
import time
from pathlib import Path
from typing import TypedDict, List

import requests
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_ollama import ChatOllama
from pinecone import Pinecone, ServerlessSpec
from langgraph.graph import StateGraph, START, END

print("Imports OK")

C:\Users\T14s\AppData\Local\Temp\ipykernel_3428\4184688129.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports OK


## 3. Environment variables

In [2]:
from pathlib import Path
from dotenv import load_dotenv, dotenv_values

env_path = Path(r"C:\Users\T14s\Downloads\.env")

print("File exists:", env_path.exists())
values = dotenv_values(env_path)
print("Keys found in .env:", list(values.keys()))

load_dotenv(env_path, override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "agentic-ai-rag")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found. Compare the names printed above with PINECONE_API_KEY.")

print("Pinecone key loaded:", bool(PINECONE_API_KEY))
print("Index name:", PINECONE_INDEX_NAME)

File exists: True
Keys found in .env: ['PINECONE_API_KEY', 'PINECONE_INDEX_NAME']
Pinecone key loaded: True
Index name: agentic-ai-rag


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "agentic-ai-rag")

print("Pinecone key loaded:", bool(PINECONE_API_KEY))

Pinecone key loaded: True


## 4. Download the Agentic AI eBook

In [4]:
pdf_url = "https://konverge.ai/pdf/Ebook-Agentic-AI.pdf"
pdf_path = Path("Ebook-Agentic-AI.pdf")   

if pdf_path.exists() and pdf_path.stat().st_size > 10_000:
    print("PDF already downloaded:", pdf_path.resolve())
else:
    response = requests.get(pdf_url, timeout=60)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
    print("PDF downloaded:", pdf_path.resolve())

PDF already downloaded: C:\Users\T14s\Agentic_AI_with_RAG_Chatbot\Ebook-Agentic-AI.pdf


## 5. Load the PDF

In [5]:
loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

print("Number of pages:", len(documents))
words_per_page = [len(d.page_content.split()) for d in documents]
print("Pages with almost no text (<20 words):", [i + 1 for i, w in enumerate(words_per_page) if w < 20])
assert sum(words_per_page) > 1000, "PDF text extraction failed - the PDF may be image-only."
print(documents[1].metadata)
print(documents[1].page_content[:500])

Number of pages: 60
Pages with almost no text (<20 words): [1, 3, 60]
{'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 27.8 (Windows)', 'creationdate': '2025-01-23T14:59:49+06:30', 'moddate': '2025-01-23T14:59:51+05:30', 'title': 'Ebook - Agentic AI Updated QR code', 'source': 'Ebook-Agentic-AI.pdf', 'total_pages': 60, 'page': 1, 'page_label': '2'}
Konverge AI is a decision science firm empowering 
businesses with the transformative power of AI. 
Operating at the intersection of data, machine 
learning (ML) models, and business insights, we 
help organizations develop cutting-edge AI 
products and solutions. 
This book provides actionable insights into Agentic 
AI, combining Konverge AI’s expertise with 
contributions from Emergence AI. Emergence AI 
shared its deep knowledge in autonomous 
multi-agent orchestration, addressing challenges 


## 6. Chunk the document

Very short chunks (titles, page numbers) are removed because they match questions like "What is Agentic AI?" without containing an answer.

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=400
)

all_chunks = text_splitter.split_documents(documents)

MIN_WORDS = 25
chunks = [c for c in all_chunks if len(c.page_content.split()) >= MIN_WORDS]

seen, unique_chunks = set(), []
for c in chunks:
    if c.page_content not in seen:
        seen.add(c.page_content)
        unique_chunks.append(c)
chunks = unique_chunks


chunk_ids = [f"chunk-{i:04d}" for i in range(len(chunks))]

print("Chunks before filtering:", len(all_chunks))
print("Chunks after filtering :", len(chunks))
print("Pages covered:", len({c.metadata.get("page") for c in chunks}))
print("-" * 60)
print(chunks[0].metadata.get("page"), chunks[0].page_content[:300])

Chunks before filtering: 131
Chunks after filtering : 129
Pages covered: 57
------------------------------------------------------------
1 Konverge AI is a decision science firm empowering 
businesses with the transformative power of AI. 
Operating at the intersection of data, machine 
learning (ML) models, and business insights, we 
help organizations develop cutting-edge AI 
products and solutions. 
This book provides actionable insi


## 7. Create text embeddings

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_embedding = embedding_model.embed_query("What is Agentic AI?")
EMBED_DIM = len(test_embedding)
print("Embedding dimensions:", EMBED_DIM)
assert EMBED_DIM == 384

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions: 384


## 8. Connect to Pinecone

In [8]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [i.name for i in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Pinecone index created.")
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(2)
else:
    print("Pinecone index already exists.")

index_info = pc.describe_index(PINECONE_INDEX_NAME)
assert index_info.dimension == EMBED_DIM, (
    f"Index dimension is {index_info.dimension} but embeddings are {EMBED_DIM}. "
    "Use a new PINECONE_INDEX_NAME or delete the old index."
)
index = pc.Index(PINECONE_INDEX_NAME)
print("Pinecone connected. Index ready.")

Pinecone index already exists.
Pinecone connected. Index ready.


## 9. Store document chunks in Pinecone

This cell **clears the index first**, then stores the chunks with stable IDs. It is safe to re-run. This is what removes the duplicate title-page vectors that caused the wrong answers.

In [9]:
try:
    index.delete(delete_all=True)
    print("Old vectors deleted.")
except Exception as e:
    print("Nothing to delete (fresh index):", type(e).__name__)


vectorstore = PineconeVectorStore(index=index, embedding=embedding_model)
vectorstore.add_documents(chunks, ids=chunk_ids)

for _ in range(15):
    time.sleep(2)
    count = index.describe_index_stats().total_vector_count
    if count == len(chunks):
        break

print("Chunks sent    :", len(chunks))
print("Vectors in index:", count)
assert count == len(chunks), "Vector count does not match - wait a few seconds and re-run this cell."

Old vectors deleted.
Chunks sent    : 129
Vectors in index: 129


## 10. Test retrieval

In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

question = "What is Agentic AI?"
retrieved_docs = retriever.invoke(question)

print("Retrieved chunks:", len(retrieved_docs))
print("Unique chunks   :", len({d.page_content for d in retrieved_docs}))
for i, doc in enumerate(retrieved_docs, 1):
    print("=" * 70)
    print("Chunk:", i, "| Page:", int(doc.metadata.get("page", -1)) + 1)
    print(doc.page_content[:500])

Retrieved chunks: 5
Unique chunks   : 5
Chunk: 1 | Page: 7
In this section, we will define what Agentic AI is and, more importantly, 
what it’s not, as it’s often misunderstood. While Agentic AI promises a 
shift from reactive to proactive problem-solving, doubts remain. Is it truly 
real? Are all the claims about it accurate? 
In this section, we cover the following topics: 
                 � What is Agentic AI? 
                 � How does it stand apart from other AI ,What can it do? 
                 � What value does it bring? 
                 � 
Chunk: 2 | Page: 5
Table of Contents
06
Practical Applications of Agentic AI
05
Your Readiness for Agentic AI
04
Orchestrating Agentic AI Systems 
03
Multi-Agent Systems
01
Introduction to Agentic AI
02
Anatomy of an Agentic AI System
Chunk: 3 | Page: 17
In this section, we explore the core components of agentic AI, explaining 
how they work together to enable autonomous decision-making and 
action, on what they are classified. How do a

## 11. Define LangGraph state

In [11]:
class RAGState(TypedDict):
    question: str
    context: List[dict]
    answer: str
    confidence: float

## 12. Create the LLM

Requires Ollama running with `ollama pull llama3.2` done.

In [12]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

try:
    print("LLM test:", llm.invoke("Reply with the single word OK").content)
except Exception as e:
    raise RuntimeError(
        "Cannot reach Ollama. Start the Ollama app and run: ollama pull llama3.2"
    ) from e

LLM test: OK


## 13. Grounding prompt

In [13]:
SYSTEM_PROMPT = """
You are an AI assistant that answers questions about the Agentic AI eBook.

Rules:
1. Answer ONLY using the provided context.
2. Do not use outside knowledge.
3. Do not invent facts.
4. Do not make assumptions.
5. If the answer cannot be found in the context, respond exactly:

"I could not find this information in the provided Agentic AI eBook."

6. Keep the answer clear and concise.
7. Every factual statement must be supported by the retrieved context.

Retrieved Context:
{context}

User Question:
{question}
"""

NOT_FOUND = "I could not find this information in the provided Agentic AI eBook."
print("SYSTEM_PROMPT loaded successfully.")

SYSTEM_PROMPT loaded successfully.


## 14. Retrieval node

In [14]:
def retrieve_node(state: RAGState):
    question = state["question"]

    
    results = vectorstore.similarity_search_with_score(question, k=10)

    context, seen = [], set()
    for doc, score in results:
        if doc.page_content in seen:
            continue
        seen.add(doc.page_content)
        context.append({
            "text": doc.page_content,
            "page": int(doc.metadata.get("page", -1)) + 1,  
            "score": round(float(score), 4),
        })
        if len(context) == 5:
            break

    confidence = round(sum(c["score"] for c in context) / len(context), 4) if context else 0.0

    return {"context": context, "confidence": confidence}

## 15. Generation node

In [15]:
import re

PREFIX_PATTERN = re.compile(
    r"^\s*(according to|based on|as per|from)\s+(the\s+)?(provided\s+|given\s+|retrieved\s+)?"
    r"(context|document|text|ebook|e-book|book|excerpt|passage)[^,:]*[,:]\s*",
    re.IGNORECASE,
)

def clean_answer(text: str) -> str:
    text = PREFIX_PATTERN.sub("", text.strip())
    return text[:1].upper() + text[1:] if text else text


def generate_node(state: RAGState):
    question = state["question"]
    context = state["context"]

    if not context:
        return {"answer": NOT_FOUND}

    context_text = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}" for item in context
    )

    prompt = SYSTEM_PROMPT.format(context=context_text, question=question)
    response = llm.invoke(prompt)

    return {"answer": clean_answer(response.content)}

## 16. Build the LangGraph workflow

In [16]:
graph_builder = StateGraph(RAGState)

graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("generate", generate_node)

graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

rag_graph = graph_builder.compile()

print("LangGraph compiled successfully.")

LangGraph compiled successfully.


## 17. Test the RAG pipeline

In [17]:
def ask_rag(question: str):
    result = rag_graph.invoke({
        "question": question,
        "context": [],
        "answer": "",
        "confidence": 0.0
    })

    return {
        "answer": result["answer"],
        "context": result["context"],
        "confidence": result["confidence"]
    }

response = ask_rag("What is Agentic AI?")

print("ANSWER:")
print(response["answer"])
print("\nRETRIEVAL SCORE (average similarity, not a probability of correctness):")
print(response["confidence"])

ANSWER:
Agentic AI refers to systems capable of autonomous decision-making and action in pursuit of specific objectives.

RETRIEVAL SCORE (average similarity, not a probability of correctness):
0.7376


## 18. Display retrieved context

In [18]:
for i, item in enumerate(response["context"], 1):
    print("=" * 80)
    print(f"Chunk {i} | Page {item['page']} | Score {item['score']}")
    print(item["text"])

Chunk 1 | Page 7 | Score 0.7812
In this section, we will define what Agentic AI is and, more importantly, 
what it’s not, as it’s often misunderstood. While Agentic AI promises a 
shift from reactive to proactive problem-solving, doubts remain. Is it truly 
real? Are all the claims about it accurate? 
In this section, we cover the following topics: 
                 � What is Agentic AI? 
                 � How does it stand apart from other AI ,What can it do? 
                 � What value does it bring? 
                 � How are businesses using it in the real world? 
INTRODUCTION 
TO AGENTIC AI
01
Chunk 2 | Page 5 | Score 0.7522
Table of Contents
06
Practical Applications of Agentic AI
05
Your Readiness for Agentic AI
04
Orchestrating Agentic AI Systems 
03
Multi-Agent Systems
01
Introduction to Agentic AI
02
Anatomy of an Agentic AI System
Chunk 3 | Page 17 | Score 0.7377
In this section, we explore the core components of agentic AI, explaining 
how they work together to enable 

## 19. Test an out-of-scope question

In [19]:
response_oos = ask_rag("Who is the current Prime Minister of India?")
print(response_oos["answer"])

I could not find this information in the provided Agentic AI eBook.


## 20. Sample queries

In [20]:
sample_questions = [
    "What is Agentic AI?",
    "What are AI agents?",
    "What are the characteristics of Agentic AI?",
    "How do AI agents differ from traditional AI?",
    "What are the components of an AI agent?",
    "What are the applications of Agentic AI?"
]

for q in sample_questions:
    print("=" * 80)
    print("QUESTION:", q)
    result = ask_rag(q)
    print("\nANSWER:")
    print(result["answer"])
    print("\nPAGES USED:", sorted({c["page"] for c in result["context"]}))
    print("RETRIEVAL SCORE:", result["confidence"])

QUESTION: What is Agentic AI?

ANSWER:
Agentic AI refers to systems capable of autonomous decision-making and action in pursuit of specific objectives.

PAGES USED: [5, 6, 7, 17, 18]
RETRIEVAL SCORE: 0.7377
QUESTION: What are AI agents?

ANSWER:
According to Page 18, "Hewitt et al. introduced actors as self-contained, interactive objects with internal states, capable of concurrent actions and communication via message-passing."

PAGES USED: [5, 6, 7, 17, 18]
RETRIEVAL SCORE: 0.7269
QUESTION: What are the characteristics of Agentic AI?

ANSWER:
The characteristics of Agentic AI are not explicitly listed. However, the context does provide information on the topics covered in the section, which includes:

* The Building Blocks of Agentic AI
* Key Components: Perception, Reasoning, Planning, Learning, and Execution
* Types and Categories of Agents
* Applications Across Industries

It also mentions that Agentic AI refers to systems capable of autonomous decision-making and action in pursuit

## 21. FastAPI application

In [21]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(
    title="Agentic AI RAG Chatbot",
    description="RAG chatbot grounded in the Agentic AI eBook",
    version="1.0.0"
)

class ChatRequest(BaseModel):
    question: str = Field(..., min_length=1)


@app.get("/")
def root():
    return {"message": "Agentic AI RAG Chatbot is running. Open /docs to test it."}


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "service": "Agentic AI RAG Chatbot"
    }


@app.post("/chat")
def chat(request: ChatRequest):
    try:
        result = ask_rag(request.question)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"RAG pipeline error: {e}")

    return {
        "answer": result["answer"],
        "context": result["context"],
        "confidence": result["confidence"]
    }

print("FastAPI app created successfully")

FastAPI app created successfully


## 22. Run FastAPI from Jupyter

Run this cell **last**. It keeps running (that is normal). Open http://127.0.0.1:8000/docs to test `/chat`.

To stop it, click **Interrupt**. Before re-running this cell, always interrupt the old one first, otherwise port 8000 is busy.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn

config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000
)

server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [3428]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:64237 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:64237 - "GET /openapi.json HTTP/1.1" 200 OK


## Architecture

PDF → PyPDFLoader → RecursiveCharacterTextSplitter → Hugging Face Embeddings → Pinecone → LangGraph Retriever → Grounded LLM → FastAPI

The LLM is instructed to use only retrieved context from the Agentic AI eBook. If the context does not contain the answer, it returns the fallback response.

The API returns:
- `answer`: final grounded answer
- `context`: retrieved chunks with page number and similarity score
- `confidence`: average retrieval score

The retrieval score is not a calibrated probability of answer correctness.